# Beatbox CNN Training
Trains a micro CNN on 200ms percussion samples and exports a TensorFlow.js model.

**Steps:**
1. Install deps
2. Upload your `beatbox_samples.zip`
3. Run all cells
4. Download `new_model.zip` at the end

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install librosa tensorflowjs scikit-learn -q

In [ ]:
# ── 2. Upload samples ZIP ─────────────────────────────────────────────────────
# Click 'Choose Files' and upload your beatbox_samples.zip exported from data_collector.html
from google.colab import files
import zipfile, os

uploaded = files.upload()   # upload beatbox_samples.zip here

zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('samples')

# Show sample counts per class
for cls in sorted(os.listdir('samples')):
    path = os.path.join('samples', cls)
    if os.path.isdir(path):
        count = len([f for f in os.listdir(path) if f.endswith('.wav')])
        print(f'  {cls}: {count} samples')

In [ ]:
# ── 3. Config & imports ───────────────────────────────────────────────────────
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
import json, shutil

SAMPLE_RATE = 44100
DURATION_MS = 200
N_SAMPLES   = int(SAMPLE_RATE * DURATION_MS / 1000)  # 6615
N_MELS      = 64
N_FFT       = 1024
HOP_LENGTH  = 256
CLASSES     = ['Background', 'Hihat', 'Kick', 'Snare']  # alphabetical = model output order
SAMPLES_DIR = 'samples'
MODEL_OUT   = 'new_model'

print('TensorFlow:', tf.__version__)

In [ ]:
# ── 4. Load & preprocess ──────────────────────────────────────────────────────
DB_FLOOR = -50.0  # dB below peak that maps to 0 brightness
                  # -50 clips ambient room noise (~-60 dB) to black while keeping percussion detail

def wav_to_mel(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=DURATION_MS / 1000)
    if len(audio) < N_SAMPLES:
        audio = np.pad(audio, (0, N_SAMPLES - len(audio)))
    else:
        audio = audio[:N_SAMPLES]

    # Peak-normalize ONLY loud sounds (real percussion hits).
    # Background (peak ~0.001) stays unnormalized so it remains quiet in dB space.
    peak = np.max(np.abs(audio))
    if peak > 0.05:
        audio = audio / peak

    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    # ref=1.0 preserves ABSOLUTE loudness — background stays dark, percussion stays bright.
    mel_db = librosa.power_to_db(mel, ref=1.0)
    mel_db = np.maximum(mel_db, DB_FLOOR)
    return (mel_db - DB_FLOOR) / (-DB_FLOOR)  # map [DB_FLOOR, 0] → [0, 1]

X, y = [], []
skipped_bg = 0
for label_idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(SAMPLES_DIR, cls)
    if not os.path.isdir(cls_dir):
        print(f'WARNING: {cls_dir} not found')
        continue
    wavs = sorted(f for f in os.listdir(cls_dir) if f.lower().endswith('.wav'))
    for fname in wavs:
        try:
            # Reject Background samples that caught an event (peak too high = not silent)
            if cls == 'Background':
                audio_peek, _ = librosa.load(os.path.join(cls_dir, fname), sr=SAMPLE_RATE, mono=True, duration=DURATION_MS/1000)
                if np.max(np.abs(audio_peek)) > 0.02:
                    skipped_bg += 1
                    continue
            X.append(wav_to_mel(os.path.join(cls_dir, fname)))
            y.append(label_idx)
        except Exception as e:
            print(f'  skip {fname}: {e}')
    loaded = (np.array(y) == label_idx).sum()
    print(f'{cls}: {loaded} loaded')

if skipped_bg:
    print(f'\n{skipped_bg} Background samples rejected (peak > 0.02 — captured an event)')

X = np.array(X, dtype=np.float32)[..., np.newaxis]  # (N, 64, frames, 1)
y = np.array(y, dtype=np.int32)
print(f'\nInput shape: {X.shape[1:]}  |  Total samples: {len(y)}')

In [ ]:
# ── 4b. Diagnostic — visualise mel spectrograms per class ─────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, len(CLASSES), figsize=(4 * len(CLASSES), 6))
for col, cls in enumerate(CLASSES):
    cls_dir = os.path.join(SAMPLES_DIR, cls)
    wavs = sorted(f for f in os.listdir(cls_dir) if f.endswith('.wav'))[:2]
    for row in range(2):
        ax = axes[row][col]
        if row < len(wavs):
            mel = wav_to_mel(os.path.join(cls_dir, wavs[row]))
            ax.imshow(mel, origin='lower', aspect='auto', cmap='magma')
            ax.set_title(f'{cls} #{row+1}', fontsize=10)
        else:
            ax.set_title(f'{cls} — no sample', fontsize=10)
        ax.axis('off')
plt.suptitle('Mel spectrograms — Kick/Hihat/Snare/Background should look clearly different', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4d. Spot-check individual Background samples + detect outliers ────────────
# If some Background samples are noisy (peaks above our threshold), they get
# peak-normalized and look like percussion. That would poison the class.

bg_mask = (y == CLASSES.index('Background'))
bg_specs = X[bg_mask, ..., 0]

# Compute "brightness" of each sample = mean pixel intensity
bg_brightness = bg_specs.mean(axis=(1, 2))
print(f'Background brightness stats:')
print(f'  min={bg_brightness.min():.3f}  max={bg_brightness.max():.3f}')
print(f'  mean={bg_brightness.mean():.3f}  median={np.median(bg_brightness):.3f}')
print(f'  samples above 0.3 (suspicious): {(bg_brightness > 0.3).sum()} / {len(bg_brightness)}')
print(f'  samples above 0.5 (likely polluted): {(bg_brightness > 0.5).sum()} / {len(bg_brightness)}')

# Show the 4 brightest Background samples — these are likely noise contamination
top_idx = np.argsort(bg_brightness)[-4:][::-1]
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, idx in enumerate(top_idx):
    axes[i].imshow(bg_specs[idx], origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
    axes[i].set_title(f'Background brightest #{i+1}\n(brightness={bg_brightness[idx]:.3f})')
    axes[i].axis('off')
plt.suptitle('Brightest Background samples — these should still look mostly dark!')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4c. Class mean spectrograms + RMS/peak stats ──────────────────────────────
# If the class means look similar, your recordings aren't distinctive enough.
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4 * len(CLASSES), 4))
for i, cls in enumerate(CLASSES):
    mask = (y == i)
    mean_spec = X[mask, ..., 0].mean(axis=0)
    axes[i].imshow(mean_spec, origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
    axes[i].set_title(f'{cls} (mean of {mask.sum()})')
    axes[i].axis('off')
plt.suptitle('Class-mean spectrograms — should look VISIBLY different')
plt.tight_layout()
plt.show()

# Print per-class RMS/peak so you can see amplitude distributions
print('\nPer-class audio stats (from raw WAV):')
for cls in CLASSES:
    cls_dir = os.path.join(SAMPLES_DIR, cls)
    wavs = sorted(f for f in os.listdir(cls_dir) if f.lower().endswith('.wav'))
    rmss, peaks = [], []
    for f in wavs:
        a, _ = librosa.load(os.path.join(cls_dir, f), sr=SAMPLE_RATE, mono=True, duration=DURATION_MS/1000)
        rmss.append(np.sqrt(np.mean(a**2)))
        peaks.append(np.max(np.abs(a)))
    print(f'  {cls:12s}  RMS mean={np.mean(rmss):.3f}  peak mean={np.mean(peaks):.3f}  n={len(wavs)}')

In [ ]:
# ── 5. Train / val split ──────────────────────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

y_cat = tf.keras.utils.to_categorical(y, num_classes=len(CLASSES))
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y
)

weights = compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y)
class_weight = dict(enumerate(weights))
print(f'Train: {len(X_train)}   Val: {len(X_val)}')
print(f'Class weights: { {CLASSES[i]: round(w, 2) for i, w in class_weight.items()} }')

In [ ]:
# ── 6. Build model ────────────────────────────────────────────────────────────
input_shape = X.shape[1:]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=input_shape),

    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(len(CLASSES), activation='softmax'),
], name='beatbox_cnn')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── 7. Train ──────────────────────────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(patience=6, factor=0.5, verbose=1),
]

history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weight,
)

In [ ]:
# ── 8. Evaluate ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f'Validation accuracy: {val_acc:.1%}')

# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_pred = model.predict(X_val).argmax(axis=1)
y_true = y_val.argmax(axis=1)
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap='Blues')
plt.title(f'Validation accuracy: {val_acc:.1%}')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9. Export to TF.js ────────────────────────────────────────────────────────
import tensorflowjs as tfjs

os.makedirs(MODEL_OUT, exist_ok=True)
tfjs.converters.save_keras_model(model, MODEL_OUT)

# Save preprocessing metadata so the browser engine knows exact params
meta = {
    'words': CLASSES,
    'sample_rate': SAMPLE_RATE,
    'duration_ms': DURATION_MS,
    'n_mels': N_MELS,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'input_shape': list(input_shape),
}
with open(os.path.join(MODEL_OUT, 'metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to', MODEL_OUT)
print('Files:', os.listdir(MODEL_OUT))

In [ ]:
# ── 10. Download ──────────────────────────────────────────────────────────────
shutil.make_archive('new_model', 'zip', MODEL_OUT)
files.download('new_model.zip')
print('Done! Unzip new_model.zip into your project folder.')